# Lesson 07 - Mẫu Thiết Kế Lập Kế Hoạch

Notebook này trình bày **Mẫu Thiết Kế Lập Kế Hoạch** cho các tác nhân AI sử dụng Microsoft Agent Framework.
Bạn sẽ học cách phân tách các yêu cầu du lịch phức tạp thành các công việc phụ có cấu trúc, phân công chúng cho các tác nhân chuyên môn,
và thực thi kế hoạch thu được — tất cả đều sử dụng đầu ra có cấu trúc được hỗ trợ bởi các mô hình Pydantic.


## Cài đặt


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Phân rã nhiệm vụ

Phân rã nhiệm vụ là cốt lõi của mẫu thiết kế lập kế hoạch. Thay vì yêu cầu một tác nhân duy nhất xử lý một yêu cầu phức tạp từ đầu đến cuối, chúng ta phân chia vấn đề thành các **nhiệm vụ con** nhỏ hơn, được xác định rõ ràng. Mỗi nhiệm vụ con được giao cho một tác nhân chuyên môn (ví dụ: chuyến bay, khách sạn, hoạt động) với ưu tiên và thứ tự phụ thuộc rõ ràng.

Cách tiếp cận này mang lại một số lợi ích:
- **Rõ ràng**: mỗi nhiệm vụ con chịu trách nhiệm duy nhất.
- **Song song**: các nhiệm vụ con độc lập có thể chạy đồng thời.
- **Đáng tin cậy**: lỗi được cô lập trong từng nhiệm vụ con.
- **Theo dõi ngân sách**: chi phí được ước tính cho từng nhiệm vụ con và tổng hợp lại.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Tạo một đại lý lập kế hoạch với đầu ra có cấu trúc

Đại lý lập kế hoạch đóng vai trò như một **người điều phối quầy lễ tân**. Dựa trên yêu cầu du lịch cấp cao, nó sẽ tạo ra một `TravelPlan` có cấu trúc — phân tách yêu cầu thành các nhiệm vụ con, đặt ưu tiên và xác định các phụ thuộc để người phục vụ hoặc lớp thực thi có thể thực hiện công việc.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Thực thi Kế hoạch bằng Công cụ Chuyên môn

Khi nhân viên lễ tân đã tạo ra một kế hoạch có cấu trúc, **đại lý hỗ trợ** sẽ thực thi kế hoạch đó.  
Mỗi công cụ chuyên môn xử lý một loại nhiệm vụ phụ (chuyến bay, khách sạn, hoạt động). Đại lý hỗ trợ  
lặp qua các nhiệm vụ phụ của kế hoạch theo thứ tự phụ thuộc và gửi từng nhiệm vụ tới công cụ  
phù hợp.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Summary

Trong bài học này bạn đã học được **Mẫu Thiết Kế Lập Kế Hoạch** cho các tác nhân AI:

1. **Phân rã nhiệm vụ** — Một tác nhân lập kế hoạch quầy tiếp tân chia yêu cầu du lịch phức tạp thành các nhiệm vụ con có cấu trúc sử dụng các mô hình Pydantic, phân công mỗi nhiệm vụ cho một tác nhân chuyên gia với các ưu tiên và phụ thuộc.
2. **Đầu ra có cấu trúc** — Bằng cách truyền `response_format`, tác nhân trả về một đối tượng `TravelPlan` đã được xác thực thay vì văn bản tự do, giúp việc xử lý tiếp theo trở nên đáng tin cậy.
3. **Thực thi kế hoạch** — Một tác nhân tiếp tân duyệt qua các nhiệm vụ con sử dụng các công cụ chuyên gia (`book_flight`, `reserve_hotel`, `book_activity`) để thực hiện kế hoạch và báo cáo kết quả.

Mẫu này tách biệt *việc phải làm* (lập kế hoạch) khỏi *cách thực hiện* (thực thi), giúp các tác nhân trở nên mô-đun hơn, dễ kiểm thử và dễ mở rộng hơn.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố từ chối trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sự không chính xác. Tài liệu gốc bằng ngôn ngữ nguyên bản nên được xem là nguồn chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp do con người thực hiện. Chúng tôi không chịu trách nhiệm cho bất kỳ sự hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
